# Нормализация зарплат

Запустите **три ячейки по порядку**: загрузка csv, функции обработки зарплат, применение к df

В `df_normalized` сохраняются **все исходные колонки + ровно две новые**:
- `salary_month_gross_normalized` — месячный эквивалент, ₽ до налога;
- `salary_hour_gross_normalized` — часовой эквивалент, ₽ до налога.



## Принятые правила (кратко, в самом конце подробнее)

| Исходное значение | Обработка |
|---|---|
| Сдельно / только «до X» | Исключаем из итоговой таблицы
| «От X» | Берём X — опубликованную нижнюю границу. |
| «X–Y» | Берём середину вилки, после распределения по вариантам часов, если оно нужно. |
| Точная сумма | Используем сумму. |
| Период отсутствует | Считаем месяцем **по согласованному допущению**|
| Налоги не указаны | Считаем gross по согласованному допущению. |
| Сумма net | Сначала приводим к месяцу, затем восстанавливаем gross по годовой прогрессивной шкале. |
| За смену / день | Делим на длительность из «Количество рабочих часов в день», умножаем на `HOURS_PER_MONTH`. |
| За час | Умножаем на `HOURS_PER_MONTH`. |
| За неделю | Умножаем на 52/12. |
| За месяц | Сохраняем месячную сумму; часовая = месячная / `HOURS_PER_MONTH`. |
| Несколько длительностей смены / дня | Одна новая строка на вариант; исходные поля повторяются без изменения. |
| Часы отсутствуют | Удаляем / Среднее из одиночных закрытых длительностей внутри профессии × графика|



**Пример разбиения:** 3 000–6 000 ₽ за смену и варианты 4–6 / 6–8 / 8–10 часов →
подвилки 3 000–4 000 / 4 000–5 000 / 5 000–6 000 ₽.
Используем их середины 3 500 / 4 500 / 5 500 ₽ и длительности 5 / 7 / 9 часов.
Это сценарное соответствие «больше часов — выше сумма», а не извлечённая из объявления связь.
Для точной суммы и «от» одну и ту же сумму используем с каждым вариантом часов.
Для месячной и недельной оплаты варианты часов не размножаем: применяем общий эквивалент нагрузки.

**НДФЛ:** стандартная годовая шкала основной базы: до 2,4 млн ₽ — 13%;
часть от 2,4 до 5 млн — 15%; от 5 до 20 млн — 18%; от 20 до 50 млн — 20%; свыше 50 млн — 22%.

Для net предполагаем одинаковую месячную сумму 12 месяцев; восстанавливаем годовую gross,
делим её на 12, затем на месячное число часов. Учитывается повышенная ставка только на превышение.



## Достаем датасет

In [31]:
# загружаем датасет

from pathlib import Path
import pandas as pd
from collections import Counter
from dataclasses import dataclass
import math
import re
import numpy as np
import pandas as pd

HOURS_PER_MONTH = 176.0
UNSPECIFIED_AS_MONTH = True # считаем неуказанный период за месячный, False - не берем в расчет
SPLIT_RANGE_BY_HOURS = True # если в смене указаны несколько периодов, то бинируем и делаем N вакансий - каждая на свое кол-во часов
IMPUTE_MISSING_HOURS = False # только для смены/оплаты в час - вставляем среднее по похожим вакансиям, False - не берем в расчет


INPUT_FILE = Path("vacancies.csv")
df_raw = pd.read_csv(INPUT_FILE, dtype=str)

# фильтр на удаленку и вахту
df_raw = df_raw[(df_raw['График'] != "Вахта") & (df_raw['Формат работы'] != "Удаленно" )]
print(f"Загружено вакансий: {len(df_raw)}")
print(f"Исходных колонок: {len(df_raw.columns)}")


Загружено вакансий: 18356
Исходных колонок: 127


## Нормализация

In [32]:
# нормализация

TARGET_COLUMNS = ["salary_month_gross_normalized", "salary_hour_gross_normalized"]
HOURS_COLUMN = "Количество рабочих часов в день"

# годовая стандартная шкала основной налоговой базы.
TAX_BRACKETS = [
    (2_400_000, 0.13),
    (5_000_000, 0.15),
    (20_000_000, 0.18),
    (50_000_000, 0.20),
    (math.inf, 0.22),
]

UNITS = {
    "за смену": "shift",
    "в смену": "shift",
    "в час": "hour",
    "за час": "hour",
    "в неделю": "week",
    "за неделю": "week",
    "за день": "day",
    "в день": "day",
    "в месяц": "month",
    "за месяц": "month",
    "месяц": "month",
    "сдельно": "piecework",
}

SALARY_RE = re.compile(
    r"(?:(от|до)\s*)?(\d[\d ]*(?:[.,]\d{1,2})?)"
    r"(?:\s*(?:[—–-]|до)\s*(\d[\d ]*(?:[.,]\d{1,2})?))?"
    r"\s*(?:₽|руб\.?|рублей|rub)"
    r"(?:\s+(" + "|".join(UNITS) + r"))?"
    r"(?:\s+(на руки|до вычета налогов|net|gross))?"
)


@dataclass(frozen=True)
class SalaryConfig:
    hours_per_month: float = 176.0
    weeks_per_year: float = 52.0
    unspecified_as_month: bool = True
    split_range_by_hours: bool = True
    impute_missing_hours: bool = True


def _clean_text(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).replace("\ufeff", " ")).strip().casefold()


def _parse_salary(value):
    text = _clean_text(value)
    if not text or text in {"зарплата не указана", "не указана"}:
        return None, "зарплата не указана"

    if "сдельн" in text and SALARY_RE.fullmatch(text) is None:
        return None, "сдельная оплата"

    match = SALARY_RE.fullmatch(text)
    if match is None:
        return None, "формат зарплаты не распознан"

    bound, first, second, unit_text, tax_text = match.groups()
    first = float(first.replace(" ", "").replace(",", "."))
    second = float(second.replace(" ", "").replace(",", ".")) if second else None

    if first <= 0 or not math.isfinite(first):
        return None, "некорректная зарплата"
    if second is not None and (second < first or not math.isfinite(second)):
        return None, "некорректная вилка"

    unit = UNITS.get(unit_text, "unspecified")
    tax = "net" if tax_text in {"на руки", "net"} else "gross"

    if unit == "piecework":
        return None, "сдельная оплата"
    if bound == "до":
        return None, "только верхняя граница «до»"
    if second is not None:
        shape, low, high = "range", first, second
    elif bound == "от":
        shape, low, high = "from", first, np.nan
    else:
        shape, low, high = "exact", first, first

    return {
        "low": low,
        "high": high,
        "shape": shape,
        "unit": unit,
        "tax": tax,
    }, None


def _parse_hours(value):
    """Возвращает список середин закрытых интервалов часов."""
    text = _clean_text(value)
    if not text:
        return [], "часы не указаны"

    result = []
    for token in dict.fromkeys(re.split(r"\s*[,;]\s*", text)):
        token = re.sub(r"\s*(?:часа|часов|час|ч\.)$", "", token).strip()
        interval = re.fullmatch(
            r"(?:от\s*)?(\d+(?:\.\d+)?)\s*(?:[–—-]|до)\s*(\d+(?:\.\d+)?)",
            token,
        )
        exact = re.fullmatch(r"\d+(?:\.\d+)?", token)
        open_interval = re.fullmatch(r"(?:до|меньше|больше|от)\s*\d+(?:\.\d+)?", token)

        if interval:
            low, high = map(float, interval.groups())
            if not 0 < low <= high <= 24:
                return [], "некорректные часы"
            result.append((low + high) / 2)
        elif exact:
            hours = float(token)
            if not 0 < hours <= 24:
                return [], "некорректные часы"
            result.append(hours)
        elif open_interval:
            # открытый интервал
            result.append(np.nan)
        else:
            return [], "формат часов не распознан"

    result.sort(key=lambda value: math.inf if pd.isna(value) else value)
    return result, None


def _fit_hours_lookup(raw):
    """средние часы с использованием полей города и профессии"""
    donors = []
    for row in raw.to_dict("records"):
        salary, _ = _parse_salary(row.get("salary_raw"))
        hours, hours_error = _parse_hours(row.get(HOURS_COLUMN))
        city = _clean_text(row.get("city_resolved"))
        profession = _clean_text(row.get("profession_resolved"))
        schedule = _clean_text(row.get("График"))

        if (
            salary is not None
            and salary["unit"] in {"shift", "day"}
            and hours_error is None
            and len(hours) == 1
            and pd.notna(hours[0])
            and profession
        ):
            donors.append((city, profession, schedule, hours[0]))

    if not donors:
        return {"city_profession_schedule": {}, "profession_schedule": {}, "profession": {}}

    table = pd.DataFrame(
        donors,
        columns=["city_resolved", "profession_resolved", "График", "hours"],
    )
    return {
        "city_profession_schedule": (
            table.loc[table["city_resolved"].ne("")]
            .groupby(["city_resolved", "profession_resolved", "График"])["hours"]
            .mean()
            .to_dict()
        ),
        "profession_schedule": (
            table.groupby(["profession_resolved", "График"])["hours"]
            .mean()
            .to_dict()
        ),
        "profession": table.groupby("profession_resolved")["hours"].mean().to_dict(),
    }


def _find_imputed_hours(row, lookup):
    """Ищет часы от более точной resolved-группы к более широкой."""
    city = _clean_text(row.get("city_resolved"))
    profession = _clean_text(row.get("profession_resolved"))
    schedule = _clean_text(row.get("График"))

    if not profession:
        return None

    return (
        lookup["city_profession_schedule"].get((city, profession, schedule))
        or lookup["profession_schedule"].get((profession, schedule))
        or lookup["profession"].get(profession)
    )


def _annual_net_to_gross(annual_net):
    """восстановление gross из годовой суммы net по прогрессивной шкале."""
    remaining = float(annual_net)
    gross = 0.0
    lower = 0.0

    for upper, rate in TAX_BRACKETS:
        net_capacity = (upper - lower) * (1 - rate)
        if remaining <= net_capacity:
            return gross + remaining / (1 - rate)
        gross += upper - lower
        remaining -= net_capacity
        lower = upper

    return gross


def normalize_salaries(raw, config=None, print_stats=True):
    """Возвращает преобразованный DataFrame: исходные колонки + две зарплаты.

    В результат входят только строки, для которых рассчитаны обе зарплаты.
    При нескольких вариантах часов одна исходная вакансия может дать несколько строк.
    """
    config = config or SalaryConfig()

    required = [
        "salary_raw",
        HOURS_COLUMN,
        "city_resolved",
        "profession_resolved",
        "График",
    ]
    missing_columns = [column for column in required if column not in raw.columns]
    if missing_columns:
        raise ValueError(f"В DataFrame отсутствуют колонки: {missing_columns}")
    if not raw.columns.is_unique:
        raise ValueError("В DataFrame повторяются названия колонок")
    if any(column in raw.columns for column in TARGET_COLUMNS):
        raise ValueError("Передайте исходный DataFrame без нормализованных колонок")
    if config.hours_per_month <= 0 or config.weeks_per_year <= 0:
        raise ValueError("Коэффициенты времени должны быть положительными")

    hours_lookup = _fit_hours_lookup(raw) if config.impute_missing_hours else {}
    output_positions = []
    normalized_values = []
    excluded = Counter()
    skipped_hour_variants = 0
    included_source_rows = set()

    for position, row in enumerate(raw.to_dict("records")):
        salary, error = _parse_salary(row.get("salary_raw"))
        if salary is None:
            excluded[error] += 1
            continue

        unit = salary["unit"]
        if unit == "unspecified":
            if config.unspecified_as_month:
                unit = "month"
            else:
                excluded["период зарплаты не указан"] += 1
                continue

        hours_options = [np.nan]
        if unit in {"shift", "day"}:
            hours_options, hours_error = _parse_hours(row.get(HOURS_COLUMN))
            if hours_error == "часы не указаны" and config.impute_missing_hours:
                imputed = _find_imputed_hours(row, hours_lookup)
                hours_options = [imputed] if imputed is not None else []
                hours_error = None if imputed is not None else "нет часов и среднего по группе"

            if hours_error is not None or not hours_options:
                excluded[hours_error or "часы не найдены"] += 1
                continue

        option_count = len(hours_options)
        valid_for_source = 0

        for option_index, hours in enumerate(hours_options):
            if unit in {"shift", "day"} and pd.isna(hours):
                skipped_hour_variants += 1
                continue

            low = salary["low"]
            high = salary["high"]

            if (
                salary["shape"] == "range"
                and option_count > 1
                and config.split_range_by_hours
            ):
                width = (high - low) / option_count
                low = low + width * option_index
                high = low + width

            representative = (
                (low + high) / 2
                if salary["shape"] == "range"
                else low
            )

            if unit in {"shift", "day"}:
                monthly = representative / hours * config.hours_per_month
            elif unit == "hour":
                monthly = representative * config.hours_per_month
            elif unit == "week":
                monthly = representative * config.weeks_per_year / 12
            elif unit == "month":
                monthly = representative
            else:
                continue

            if salary["tax"] == "net":
                monthly = _annual_net_to_gross(monthly * 12) / 12

            hourly = monthly / config.hours_per_month
            output_positions.append(position)
            normalized_values.append((monthly, hourly))
            valid_for_source += 1

        if valid_for_source:
            included_source_rows.add(position)
        else:
            excluded["нет пригодного варианта часов"] += 1

    result = raw.iloc[output_positions].copy().reset_index(drop=True)
    result[TARGET_COLUMNS] = pd.DataFrame(
        normalized_values,
        columns=TARGET_COLUMNS,
        dtype=float,
    )

    if print_stats:
        print(f"Исходных вакансий: {len(raw):,}")
        print(f"Нормализовано исходных вакансий: {len(included_source_rows):,}")
        print(f"Исключено исходных вакансий: {len(raw) - len(included_source_rows):,}")
        print(f"Строк в результате с учётом вариантов часов: {len(result):,}")
        if excluded:
            print("\nПричины исключения:")
            for reason, count in excluded.most_common():
                print(f"  {reason}: {count:,}")

    return result

## Применение

In [33]:
# применение функции к загруженному DataFrame

# class SalaryConfig:
#     hours_per_month: float = 176.0
#     weeks_per_year: float = 52.0
# UNSPECIFIED_AS_MONTH = True  считаем неуказанный период за месячный, False - не берем в расчет
# SPLIT_RANGE_BY_HOURS = True  если в смене указаны несколько периодов, то бинируем и делаем N вакансий - каждая на свое кол-во часов
# IMPUTE_MISSING_HOURS = False  только для смены/оплаты в час - вставляем среднее по похожим вакансиям, False - не берем в расчет

config = SalaryConfig(
    hours_per_month=HOURS_PER_MONTH,
    unspecified_as_month=True,
    split_range_by_hours=True,
    impute_missing_hours=False,
)


df_normalized = normalize_salaries(df_raw, config=config)

print("\nПример результата:")
display(df_normalized[["city_resolved", "profession_resolved", "salary_raw", "salary_month_gross_normalized", "salary_hour_gross_normalized",]].head(15))


Исходных вакансий: 18,356
Нормализовано исходных вакансий: 14,357
Исключено исходных вакансий: 3,999
Строк в результате с учётом вариантов часов: 19,554

Причины исключения:
  только верхняя граница «до»: 2,131
  часы не указаны: 1,120
  зарплата не указана: 499
  сдельная оплата: 199
  нет пригодного варианта часов: 45
  некорректная зарплата: 5

Пример результата:


,city_resolved,profession_resolved,salary_raw,salary_month_gross_normalized,salary_hour_gross_normalized
0,Санкт-Петербург,NaN,55 000 — 65 000 ₽ \nна руки,68965.517241,391.849530
1,Санкт-Петербург,Водитель грузового транспорта,от 120 000 ₽,120000.000000,681.818182
2,Санкт-Петербург,Продавец,2 500 — 3 000 ₽ \nза смену на руки,48375.812094,274.862569
3,Крымск,Грузчик,55 000 — 70 000 ₽,62500.000000,355.113636
4,Набережные Челны,Грузчик,76 000 — 80 000 ₽ \nна руки,89655.172414,509.404389
5,Екатеринбург,Грузчик,от 400 ₽ \nв час на руки,80919.540230,459.770115
6,Санкт-Петербург,Водитель грузового транспорта,180 000 — 220 000 ₽,200000.000000,1136.363636
7,Москва,NaN,от 5 000 ₽ \nза смену на руки,106473.079250,604.960678
8,Москва,NaN,от 5 000 ₽ \nза смену на руки,87956.021989,499.750125
9,Екатеринбург,Продавец,30 000 — 60 000 ₽,45000.000000,255.681818


# Правила нормализации опубликованных зарплат, версия 1


## Результат

- Функция принимает исходный DataFrame вакансий.
- В результат включаются только вакансии, для которых рассчитаны обе нормализованные суммы.
- Все исходные колонки сохраняются.
- Добавляются две колонки:
  - `salary_month_gross_normalized` — нормализованная зарплата gross в месяц;
  - `salary_hour_gross_normalized` — нормализованная зарплата gross в час.
- Часовая сумма рассчитывается как месячная сумма, делённая на `hours_per_month`.
- По умолчанию `hours_per_month = 176`.
- При нескольких вариантах часов одна исходная вакансия может создать несколько строк.
- Отдельный аудит преобразований не возвращается.

## Форма опубликованной зарплаты

- **Точная сумма (`exact`)**:
  - используется опубликованное значение;
  - нижняя и верхняя границы считаются равными.
- **Нижняя граница (`from`)**:
  - используется опубликованная нижняя граница;
  - значение считается числовым нижним ориентиром;
  - результат не считается точной зарплатой.
- **Закрытая вилка (`range`)**:
  - при одном варианте часов или отсутствии необходимости учитывать часы используется середина всей вилки;
  - при нескольких вариантах часов для сменной или дневной оплаты вилка может быть разделена на равные части;
  - середина вилки или подвилки считается числовым ориентиром, а не наблюдаемой точной выплатой.
- **Верхняя граница (`to`)**:
  - вакансия исключается;
  - верхняя граница не используется как точечная оценка.
- **Отсутствующая зарплата**:
  - вакансия исключается;
  - зарплата по похожим вакансиям не подбирается.
- **Нулевая или отрицательная сумма**:
  - вакансия исключается как некорректная.
- **Перевёрнутая или некорректная вилка**:
  - вакансия исключается.
- **Нераспознанный формат**:
  - вакансия исключается.

## Оплата за месяц

- Распознаются единицы `в месяц`, `за месяц`, `месяц`.
- Для точной суммы месячная нормализованная зарплата равна опубликованной сумме.
- Для формата `from` месячная сумма равна нижней границе.
- Для формата `range` месячная сумма равна середине вилки.
- Формат `to` исключается.
- Часовая нормализованная зарплата рассчитывается как месячная сумма, делённая на `hours_per_month`.
- Поле `Количество рабочих часов в день` не используется.
- Часовая сумма является стандартизованным эквивалентом, а не фактической ставкой по опубликованному графику.

## Оплата за неделю

- Распознаются единицы `в неделю`, `за неделю`.
- Для точной суммы используется опубликованная недельная сумма.
- Для формата `from` используется нижняя недельная граница.
- Для формата `range` используется середина недельной вилки.
- Формат `to` исключается.
- Месячная нормализованная зарплата рассчитывается как недельная сумма, умноженная на `weeks_per_year` и делённая на 12.
- По умолчанию `weeks_per_year = 52`.
- Часовая нормализованная зарплата рассчитывается как месячная сумма, делённая на `hours_per_month`.
- Поле `Количество рабочих часов в день` не используется.
- Фактическое количество часов в неделю не рассчитывается.
- Часовая сумма является стандартизованным эквивалентом при заданном `hours_per_month`.

## Оплата за час

- Распознаются единицы `в час`, `за час`.
- Для точной суммы используется опубликованная часовая ставка.
- Для формата `from` используется нижняя часовая граница.
- Для формата `range` используется середина часовой вилки.
- Формат `to` исключается.
- Месячная нормализованная зарплата рассчитывается как часовая сумма, умноженная на `hours_per_month`.
- После налоговой нормализации часовая gross-сумма рассчитывается как месячная gross-сумма, делённая на `hours_per_month`.
- Поле `Количество рабочих часов в день` не используется.
- Месячная сумма предполагает нагрузку `hours_per_month` оплачиваемых часов.

## Оплата за смену

- Распознаются единицы `за смену`, `в смену`.
- Для расчёта требуется продолжительность смены.
- Для точной суммы используется опубликованная сумма за смену.
- Для формата `from` используется нижняя граница суммы за смену.
- Для формата `range` при одном варианте часов используется середина всей вилки.
- Формат `to` исключается.
- Часовая ставка до налоговой нормализации рассчитывается как сумма за смену, делённая на продолжительность смены.
- Месячная сумма рассчитывается как часовая ставка, умноженная на `hours_per_month`.
- При отсутствии часов и включённом `impute_missing_hours` выполняется подбор среднего количества часов.
- При отсутствии часов и выключенном `impute_missing_hours` вакансия исключается.
- Если часы не удалось подобрать, вакансия исключается.
- Месячная сумма является эквивалентом при заданном `hours_per_month`, а не расчётом по фактическому числу смен.

## Оплата за день

- Распознаются единицы `за день`, `в день`.
- Оплата за день обрабатывается по тем же правилам, что оплата за смену.
- Для расчёта требуется количество рабочих часов в день.
- Для точной суммы используется опубликованная дневная сумма.
- Для формата `from` используется нижняя дневная граница.
- Для формата `range` при одном варианте часов используется середина всей вилки.
- Формат `to` исключается.
- Часовая ставка до налоговой нормализации рассчитывается как дневная сумма, делённая на часы в день.
- Месячная сумма рассчитывается как часовая ставка, умноженная на `hours_per_month`.
- Отсутствующие часы обрабатываются тем же способом, что для оплаты за смену.

## Зарплата без указанного периода

- Если единица оплаты отсутствует, назначается `unspecified`.
- При `unspecified_as_month=True` сумма считается месячной.
- При `unspecified_as_month=False` вакансия исключается.
- Для точной суммы используется опубликованное значение.
- Для формата `from` используется нижняя граница.
- Для формата `range` используется середина вилки.
- Формат `to` исключается.
- При трактовке как месячной часовая сумма рассчитывается делением на `hours_per_month`.
- Поле `Частота выплат` не используется для определения периода зарплаты.
- Неуказанный период, признанный месячным, является рабочим допущением.

## Сдельная оплата

- Явная единица `сдельно` исключается.
- Нестандартная строка, содержащая основу `сдельн`, также исключается как сдельная.

## Формат рабочих часов

- Поле рабочих часов используется только для оплаты за смену и за день.
- Точное значение часов используется без изменения.
- Закрытый интервал часов заменяется его серединой.
- Несколько значений разделяются по запятым или точкам с запятой.
- Каждый пригодный вариант часов обрабатывается отдельно.
- Повторяющиеся текстовые варианты удаляются.
- Варианты сортируются по возрастанию продолжительности.
- Нулевые, отрицательные, превышающие 24 часа и перевёрнутые интервалы считаются некорректными.
- Нераспознанный формат часов приводит к исключению вакансии.

## Несколько вариантов часов и точная зарплата

- Одна и та же точная сумма применяется к каждому пригодному варианту часов.
- Для каждого варианта рассчитываются отдельные часовая и месячная суммы.
- Количество строк результата равно количеству пригодных вариантов часов.
- Точная сумма на части не делится.

## Несколько вариантов часов и зарплата `from`

- Одна и та же нижняя граница применяется к каждому пригодному варианту часов.
- Для каждого варианта рассчитываются отдельные часовая и месячная нижние оценки.
- Нижняя граница на части не делится.
- Количество строк результата равно количеству пригодных вариантов часов.

## Несколько вариантов часов и закрытая вилка

- При `split_range_by_hours=True`:
  - исходная вилка делится на равные подвилки по количеству вариантов часов;
  - варианты часов предварительно сортируются по возрастанию;
  - более короткому варианту назначается более низкая подвилка;
  - более длинному варианту назначается более высокая подвилка;
  - для расчёта используется середина назначенной подвилки;
  - связь между продолжительностью смены и частью вилки является сценарным допущением.
- При `split_range_by_hours=False`:
  - исходная вилка не делится;
  - для каждого варианта часов используется середина всей вилки;
  - количество строк результата всё равно равно количеству пригодных вариантов часов.
- Открытые варианты часов учитываются в исходном количестве частей вилки, но затем пропускаются.
- Из-за этого часть вилки, назначенная открытому варианту, может не попасть в результат.

## Подбор отсутствующих часов

- Подбор выполняется только для оплаты за смену и за день.
- Подбор выполняется только при полностью пустом поле часов.
- Подбор не выполняется для открытого или нераспознанного формата часов.
- Подбор включается параметром `impute_missing_hours=True`.
- При `impute_missing_hours=False` вакансия с отсутствующими часами исключается.
- Донорами для расчёта средних часов являются вакансии:
  - с распознанной зарплатой;
  - с оплатой за смену или день;
  - с заполненным `profession_resolved`;
  - с одним вариантом часов;
  - с точным значением часов или одним закрытым интервалом;
  - без открытого интервала часов.
- Для закрытого интервала целевым значением является его середина.
- Даже один донор может определить среднее группы.
- Каскад поиска среднего:
  - `city_resolved × profession_resolved × График`;
  - `profession_resolved × График`;
  - `profession_resolved`;

## Налоговая нормализация

- `на руки` и `net` считаются зарплатой net.
- `до вычета налогов` и `gross` считаются зарплатой gross.
- Если налоговый статус отсутствует, сумма считается gross.
- Для gross налоговый пересчёт не выполняется.
- Для net сумма сначала приводится к месячному периоду.
- Месячная net-сумма считается одинаковой на протяжении 12 месяцев.
- Годовая net-сумма восстанавливается до gross по прогрессивной шкале.
- Годовая gross-сумма делится на 12.
- Повышенная ставка применяется только к части дохода выше соответствующего порога.
- Используются ставки 13%, 15%, 18%, 20% и 22% с заданными в коде годовыми порогами.
- Налоговые вычеты не учитываются.
- Другие доходы человека не учитываются.
- Специальные налоговые режимы не учитываются.
- Статус нерезидента не учитывается.
- Особенности самозанятости и ИП не учитываются.
